# Embedding Models using HuggingFace
This notebook demonstrates embedding models using HuggingFace and related libraries

## Install Required Libraries

In [ ]:
# Install required packages
# pip install sentence-transformers transformers torch scikit-learn python-dotenv

## Import Required Libraries

In [ ]:
from dotenv import load_dotenv
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
load_dotenv()

## Method 1: Using Sentence Transformers (Recommended for Semantic Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer

# Initialize the model
# all-MiniLM-L6-v2 is a lightweight model with good performance
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Embed a single query
query = "India is a growing country"
query_embedding = model.encode(query)

print(f"Query: {query}")
print(f"Embedding shape: {query_embedding.shape}")
print(f"Embedding: {query_embedding}")

In [ ]:
# Get dimension of embedding vector
print(f"Dimension of embedding: {len(query_embedding)}")

## Method 1a: Using Different Sentence Transformer Models

In [ ]:
# Using a larger model for better quality embeddings
# all-MiniLM-L12-v2 has better performance but slightly larger
model_large = SentenceTransformer('all-MiniLM-L12-v2')
query_embedding_large = model_large.encode(query)

print(f"Dimension (all-MiniLM-L12-v2): {len(query_embedding_large)}")

In [ ]:
# Using multi-qa model for better retrieval
model_qa = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
query_embedding_qa = model_qa.encode(query)

print(f"Dimension (multi-qa-MiniLM-L6-cos-v1): {len(query_embedding_qa)}")

## Method 2: Using HuggingFace Transformers Directly

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Load model and tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
transformer_model = AutoModel.from_pretrained(model_name)

In [ ]:
# Encode query using transformers
text = "India is a growing country"
encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = transformer_model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
print(f"Embedding shape: {sentence_embeddings.shape}")
print(f"Embedding: {sentence_embeddings}")

## Method 3: Using HuggingFace Hub API

In [ ]:
# Using HuggingFace Hub API for inference (requires HF_TOKEN)
from huggingface_hub import InferenceClient

# Initialize inference client
hf_token = os.getenv('HUGGINGFACEHUB_API_KEY') or os.getenv('HF_TOKEN')
client = InferenceClient(token=hf_token)

# Get embeddings using HuggingFace Hub
text = "India is a growing country"
# Note: You need a HuggingFace token for this to work
# embeddings = client.get_model_embeddings(text)
print("HuggingFace Hub method requires authentication")

## Embedding Multiple Documents (Using Sentence Transformers)

In [ ]:
# Initialize model
model = SentenceTransformer('all-MiniLM-L6-v2')

# List of documents
documents = [
    "what is a capital of USA",
    "who is a president of usa",
    "who is a prime minister of india"
]

# Embed all documents
document_embeddings = model.encode(documents)

print(f"Number of documents: {len(document_embeddings)}")
print(f"Each embedding dimension: {len(document_embeddings[0])}")

In [ ]:
# Display first document embedding
print(f"First document embedding:\n{document_embeddings[0]}")

## Cosine Similarity Search with HuggingFace Embeddings

In [ ]:
# Query and documents
documents = [
    "what is a capital of USA",
    "who is a president of usa",
    "who is a prime minister of india"
]

my_query = "narendra modi is indian prime minister"

# Get embeddings
query_embedding = model.encode(my_query)
document_embeddings = model.encode(documents)

In [ ]:
# Calculate cosine similarity
scores = cosine_similarity([query_embedding], document_embeddings)
print(f"Cosine similarity scores:\n{scores}")

In [ ]:
# Find most similar document
most_similar_idx = np.argmax(scores[0])
similarity_score = scores[0][most_similar_idx]

print(f"\nQuery: {my_query}")
print(f"\nMost similar document: {documents[most_similar_idx]}")
print(f"Similarity score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")

## Comparison of Different HuggingFace Models

In [ ]:
# Compare different embedding models
models_to_compare = [
    'all-MiniLM-L6-v2',
    'all-MiniLM-L12-v2',
    'multi-qa-MiniLM-L6-cos-v1',
    'sentence-transformers/paraphrase-MiniLM-L6-v2'
]

results = {}
for model_name in models_to_compare:
    try:
        st_model = SentenceTransformer(model_name)
        embedding = st_model.encode("test")
        results[model_name] = len(embedding)
    except Exception as e:
        results[model_name] = f"Error: {str(e)}"

print("Model Comparison (Dimension):")
for model, dimension in results.items():
    print(f"  {model}: {dimension}")

## Advanced: Semantic Search with Multiple Documents

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Initialize model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample documents
documents = [
    "The capital of USA is Washington DC",
    "The president of USA is elected every 4 years",
    "India's prime minister leads the government",
    "New York is the largest city in USA",
    "Paris is the capital of France",
    "Tokyo is the capital of Japan"
]

# Query
query = "Who governs United States?"

# Get embeddings
query_embedding = model.encode(query)
doc_embeddings = model.encode(documents)

# Calculate similarity
similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

# Create results dataframe
results_df = pd.DataFrame({
    'Document': documents,
    'Similarity Score': similarities
})

# Sort by similarity
results_df = results_df.sort_values('Similarity Score', ascending=False)

print(f"Query: {query}\n")
print(results_df.to_string(index=False))

## Batch Encoding for Performance

In [ ]:
import time

# Large corpus of documents
large_corpus = [
    f"Document about topic {i}" for i in range(1000)
]

model = SentenceTransformer('all-MiniLM-L6-v2')

# Efficient batch encoding
start_time = time.time()
corpus_embeddings = model.encode(large_corpus, batch_size=32, show_progress_bar=True)
end_time = time.time()

print(f"\nEncoded {len(large_corpus)} documents in {end_time - start_time:.2f} seconds")
print(f"Embedding shape: {corpus_embeddings.shape}")

## Summary: HuggingFace vs LangChain Embeddings

### Advantages of Direct HuggingFace Usage:
- **More control**: Direct access to model parameters
- **Lightweight**: No LangChain wrapper overhead
- **Faster**: Direct implementation of embeddings
- **Flexible**: Easy to customize for specific needs

### Advantages of LangChain:
- **Abstraction**: Unified interface for multiple providers
- **Integration**: Easy integration with LLMs and other components
- **Convenience**: Pre-configured models and defaults

### Popular HuggingFace Models:
- `all-MiniLM-L6-v2`: Lightweight, good performance (384 dimensions)
- `all-MiniLM-L12-v2`: Better quality, slightly larger (384 dimensions)
- `multi-qa-MiniLM-L6-cos-v1`: Optimized for semantic search
- `paraphrase-MiniLM-L6-v2`: Good for paraphrase detection